# 03 · Train a wider U-Net on Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammettcepi/seviri-fog-segmentation/blob/feature/unet-colab/notebooks/03_train_unet_colab.ipynb)

**Goal:** run `fogseg train` with more channels than the 4 GB GTX 1050 Ti can hold, and compare the cross-validation
with the 16-channel model of [`docs/notes/unet.md`](../docs/notes/unet.md).

Choose a GPU first: *Runtime → Change runtime type → T4 GPU*. Run the notebook once per width, in order.

**The protocol of a wider model:**

- It is compared on the CV over night folds 1–4 only: pooled POD at FAR ≤ 0.28, from `unet_cv.csv`.
- This notebook does not run `fogseg predict` or `fogseg evaluate`. The held-out set has been looked at four times;
  a wider model's score there would be a fifth look.
- The 2024-25 test season stays closed. `fogseg train` reads only the scenes of 2023-24, and the case tables of
  2024-25 must never be copied to Drive.

**Copy the data first**, as the README's section "Training a wider U-Net on Colab" describes. `MyDrive/fogseg` holds:

```
processed/seviri_ceu.zarr
processed/labels_teacher.zarr
processed/dem_ceu.zarr
stations/cases_fixed_2023-24.csv
stations/splits_2023-24.csv
```

In [ ]:
import os

BASE_CHANNELS = 32  # @param [32, 48] {type:"raw"}
BRANCH = "feature/unet-colab"

DRIVE = "/content/drive/MyDrive/fogseg"
REPO = "/content/seviri-fog-segmentation"

os.environ["FOGSEG_BASE_CHANNELS"] = str(BASE_CHANNELS)
os.environ["FOGSEG_DRIVE"] = DRIVE

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
![ -d {REPO} ] || git clone --quiet --depth 1 --branch {BRANCH} https://github.com/muhammettcepi/seviri-fog-segmentation.git {REPO}
!pip install --quiet -e "{REPO}[model]"
!fogseg train --help | head -n 1

## Check the data on Drive

These are the files the 16-channel model was trained on. The check stops if a file differs, or if the stations folder
holds anything but these two files, so that neither a test-season table nor a held-out record can reach a run.

In [ ]:
import hashlib
import json
from pathlib import Path

INPUT_FILES = {
    "cases_fixed_2023-24.csv": "813d77c029731e23976cf90a059a9902da6ce8e616f9307f83e69863e00d9dd3",
    "splits_2023-24.csv": "217afcf23dc82fbd1ab321bae35187d40e15ce3f468de17a9fcf5a2004d31df1",
}
LABELS_CREATED = "2026-11-07T10:15:19+00:00"

input_stations = Path(DRIVE) / "stations"
processed = Path(DRIVE) / "processed"

found = {path.name for path in input_stations.iterdir()}
if found != set(INPUT_FILES):
    raise RuntimeError(f"{input_stations} must hold only {sorted(INPUT_FILES)}, but holds {sorted(found)}")
for name, expected in INPUT_FILES.items():
    if hashlib.sha256((input_stations / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"{name} differs from the file the 16-channel model was trained on")
for store in ("seviri_ceu.zarr", "labels_teacher.zarr", "dem_ceu.zarr"):
    if not (processed / store / "zarr.json").exists():
        raise FileNotFoundError(f"{processed / store} is missing or incomplete")
labels = json.loads((processed / "labels_teacher.zarr" / "zarr.json").read_text())
if labels["attributes"]["created"] != LABELS_CREATED:
    raise RuntimeError("labels_teacher.zarr is not the store the 16-channel model was trained on")
print("The data on Drive are the data of the 16-channel model")

## Write the config of this run

Every width gets its own folder `runs/unet_base<width>` with its config, log, model and its own stations folder, so a
run never overwrites another one. Only `unet.base_channels` differs from the 16-channel training; the epochs, batch
size and learning rate stay as they are, or the CV would not compare widths alone.

In [ ]:
import shutil

import yaml


def on_drive(value):
    """The config value with its local data paths moved to Drive."""
    if isinstance(value, dict):
        return {key: on_drive(item) for key, item in value.items()}
    if isinstance(value, str):
        return value.replace("/mnt/d/fogseg-data", DRIVE)
    return value


run = Path(DRIVE) / "runs" / f"unet_base{BASE_CHANNELS}"
stations = run / "stations"
if (stations / "unet_cv.csv").exists():
    raise FileExistsError(f"{run} already holds a finished run; choose another width or move it away")
stations.mkdir(parents=True, exist_ok=True)
(Path(DRIVE) / "mlflow").mkdir(exist_ok=True)
for name in INPUT_FILES:
    shutil.copyfile(input_stations / name, stations / name)

cfg = on_drive(yaml.safe_load(Path(REPO, "configs", "default.yaml").read_text()))
cfg["data"]["log_dir"] = str(run / "logs")
cfg["stations"]["data_dir"] = str(stations)
cfg["unet"].update(
    model_path=str(run / "unet.pt"),
    predictions_path=str(run / "predictions_unet.zarr"),
    experiment="fogseg-unet-colab",
    base_channels=BASE_CHANNELS,
    device="cuda",
)
CONFIG_PATH = run / "unet_colab.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(CONFIG_PATH)

## Will it fit, and how long will it take?

A few training steps on random scenes of the real size and batch. The estimate counts the 16 CV fits and the final fit
at 40 epochs, times 1.4 for reading, scoring and MLflow, the ratio of the 16-channel run on the GTX 1050 Ti. A free
runtime can stop after a few hours, and `fogseg train` cannot resume.

In [ ]:
%%python
import os
import time

import torch
import torch.nn.functional as F

from fogseg.unet import UNet

if not torch.cuda.is_available():
    raise SystemExit("No GPU: choose Runtime > Change runtime type > T4 GPU")
device = torch.device("cuda")
print(torch.cuda.get_device_name(device))

model = UNet(14, int(os.environ["FOGSEG_BASE_CHANNELS"]), 4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(4, 14, 241, 462, device=device)
y = torch.rand(4, 241, 462, device=device)
for step in range(30):
    if step == 10:
        torch.cuda.synchronize()
        started = time.perf_counter()
    loss = F.binary_cross_entropy_with_logits(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

seconds = (time.perf_counter() - started) / 20
steps = 16 * 40 * 47 + 40 * 62
weights = sum(p.numel() for p in model.parameters()) / 1e6
peak = torch.cuda.max_memory_allocated(device) / 2**30
total = torch.cuda.get_device_properties(device).total_memory / 2**30
print(f"{weights:.2f} M weights, peak {peak:.1f} of {total:.1f} GB GPU memory")
print(f"{seconds * 1000:.0f} ms per step, so about {1.4 * seconds * steps / 3600:.1f} h of training")

## Can MLflow write to Drive?

The runs go to a SQLite file on Drive so that they outlive the runtime. SQLite on a network file system can fail, so
this tries it on a scratch database first, rather than hours into the training.

In [ ]:
%%python
import os
import time
from pathlib import Path

import mlflow

path = Path(os.environ["FOGSEG_DRIVE"], "mlflow", "preflight.db")
mlflow.set_tracking_uri(f"sqlite:///{path}")
started = time.perf_counter()
with mlflow.start_run():
    for step in range(50):
        mlflow.log_metrics({"loss": 1.0 / (step + 1)}, step=step)
print(f"Creating the database and 50 writes took {time.perf_counter() - started:.0f} s")
path.unlink()

## Train

Four loss settings on four night folds, then the winner on all 248 scenes. Keep this tab open: the output and
`logs/train.log` in the run folder show the progress. At the end the run folder holds `stations/unet_cv.csv`,
`stations/unet_oof.csv`, `stations/unet_config.json` and `unet.pt`.

In [ ]:
!fogseg train --config {CONFIG_PATH}

## Compare with 16 channels

The pooled CV of every finished width, against the 16-channel model of `docs/notes/unet.md`. A gain counts only as far
as it holds across the folds: they spread by 0.13 POD and each width is one training run per fold.

In [ ]:
%%python
import os
from pathlib import Path

import pandas as pd

from fogseg.experiment import pooled_scores, winner

rows = [
    {
        "base": 16,
        "winner": "4 soft points 1",
        "p*": 0.40,
        "POD": 0.821,
        "FAR": 0.277,
        "CSI": 0.624,
        "POD per fold": "0.776 / 0.749 / 0.863 / 0.880",
    }
]
for path in sorted(Path(os.environ["FOGSEG_DRIVE"], "runs").glob("unet_base*/stations/unet_cv.csv")):
    table = pd.read_csv(path)
    best = winner(table)
    pooled = pooled_scores(table).loc[best]
    folds = table[(table["variant"] == best) & (table["fold"].astype(str) != "pooled")]
    rows.append(
        {
            "base": int(path.parents[1].name.removeprefix("unet_base")),
            "winner": best,
            "p*": round(float(pooled["value"]), 2),
            "POD": round(float(pooled["pod"]), 3),
            "FAR": round(float(pooled["far"]), 3),
            "CSI": round(float(pooled["csi"]), 3),
            "POD per fold": " / ".join(f"{pod:.3f}" for pod in folds["pod"]),
        }
    )
print(pd.DataFrame(rows).sort_values("base").to_string(index=False))

Finally, write everything to Drive before the runtime goes away.

In [ ]:
drive.flush_and_unmount()